# SchoolBridge — LayoutXLM Sentence Grouping PoC (image 포함)

**목적**: LiLT가 셔플 검증에서 43% 정확도로 실패 → bbox 좌표만으로는 column 학습 불가능 증명됨. **시각 신호 (image stream) 추가하면 진짜 layout-aware grouping 가능한지** 검증.

## LiLT vs LayoutXLM 차이

| | LiLT | **LayoutXLM** |
|---|---|---|
| 입력 | text + bbox | text + bbox + **image** |
| Visual signal | ❌ | ✅ (헤더 굵기·배경·정렬) |
| detectron2 | 불필요 | 필요 |
| 모델 크기 | 140M | 360M |
| Multilingual | XLM-R/InfoXLM | 자체 multilingual |

## 검증 흐름

1. 구강검진 PDF 1장 라벨링 → Mini fine-tune → 같은 페이지 추론 (B안)
2. **셔플 검증** — 추출 순서 무작위화 → bbox + image만으로 column 분리 가능한가

**Colab 세팅**: 런타임 → T4 GPU

## 1. 환경 + 설치 (~10분, detectron2 빌드 포함)

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!apt-get install -y -q fonts-nanum > /dev/null 2>&1
!fc-cache -fv > /dev/null 2>&1
print("설치 완료 — 런타임 재시작 필요할 수 있음 (UI: 런타임 → 세션 다시 시작)")

In [ ]:
import torch
import detectron2
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("detectron2:", detectron2.__version__)
print("LayoutXLM import OK")

## 2. PDF 업로드

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
pdf_paths = [Path(name) for name in uploaded.keys()
             if Path(name).suffix.lower() == ".pdf"]
for p in pdf_paths:
    print(f"  {p.name}: {p.stat().st_size // 1024} KB")

## 3. PDF에서 토큰 + bbox 추출 + 페이지 이미지 렌더링

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import pdfplumber
import fitz
from PIL import Image

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]
    page: int = 0

def extract_pdf(path: Path) -> List[TextSpan]:
    spans = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            W, H = page.width, page.height
            words = page.extract_words(
                use_text_flow=True,
                keep_blank_chars=False,
                x_tolerance=3,
                y_tolerance=3,
            )
            for w in words:
                bbox = (w["x0"]/W, w["top"]/H, w["x1"]/W, w["bottom"]/H)
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx))
    return spans

def render_pdf_page(path: Path, page_idx: int = 0, dpi: int = 150) -> Image.Image:
    doc = fitz.open(path)
    page = doc[page_idx]
    pix = page.get_pixmap(dpi=dpi)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    doc.close()
    return img

sample_pdf = pdf_paths[0]
all_spans = extract_pdf(sample_pdf)
page_spans = [s for s in all_spans if s.page == 0]
page_image = render_pdf_page(sample_pdf, page_idx=0, dpi=150)

print(f"📄 {sample_pdf.name}")
print(f"  Page 0 tokens: {len(page_spans)}")
print(f"  Page image: {page_image.size}")

## 4. LayoutXLM 모델 로드

In [ ]:
MODEL_ID = "microsoft/layoutxlm-base"
MAX_SENT_ID = 20

processor = LayoutXLMProcessor.from_pretrained(MODEL_ID, apply_ocr=False)
model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=MAX_SENT_ID,
    id2label={i: f"SENT_{i}" for i in range(MAX_SENT_ID)},
    label2id={f"SENT_{i}": i for i in range(MAX_SENT_ID)},
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params | Device: {device}")

## 5. 입력 변환 함수 (image 포함)

In [ ]:
def to_layoutxlm_inputs(image, spans, word_labels=None):
    words = [s.text for s in spans]
    boxes = []
    for s in spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([
            max(0, min(1000, int(x0 * 1000))),
            max(0, min(1000, int(y0 * 1000))),
            max(0, min(1000, int(x1 * 1000))),
            max(0, min(1000, int(y1 * 1000))),
        ])
    encoded = processor(
        image, words, boxes=boxes,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    
    # word_labels → 토큰별 labels (special token은 -100)
    if word_labels is not None:
        word_ids = encoded.word_ids()
        token_labels = []
        for word_id in word_ids:
            if word_id is None:
                token_labels.append(-100)
            else:
                token_labels.append(word_labels[word_id])
        encoded["labels"] = torch.tensor([token_labels])
    
    return encoded

print("to_layoutxlm_inputs 정의 완료 (image + text + bbox)")

## 6. 라벨링용 spans 확인 — 인덱스 + 텍스트

In [ ]:
print(f"=== {sample_pdf.name} — page_spans ({len(page_spans)}개) ===\n")
for i, s in enumerate(page_spans):
    y = s.bbox[1]
    print(f"  [{i:3d}] (y={y:.3f}) {s.text!r}")

## 7. 라벨링 — sentence_ranges

이전 LiLT 노트북과 동일한 라벨링. 위 셀 출력 보고 정확한 index로 채우기.

In [ ]:
from collections import Counter

sentence_ranges = [
    # 본문
    (0,   9,   0, "발송 정보"),
    (10,  15,  1, "통신문 제목 + 인사"),
    (16,  60,  2, "본문 안내문"),
    (61,  75,  3, "1. 검진 대상"),
    (76,  95,  4, "2. 검진 기간"),
    (96,  110, 5, "3. 검진 항목"),
    (111, 125, 6, "4. 검사 비용"),
    (126, 145, 7, "5. 검진 기관"),
    
    # ===== 표 영역 — 핵심 검증 =====
    (146, 168, 10, "진심담은치과의원 — 시간·예약·방문 (column 1)"),
    (169, 186, 11, "청담i치과의원 — 시간·예약·방문 (column 2)"),
    
    # 절취선 이후
    (187, 196, 12, "발급일자 + 가능초등학교장"),
    (197, 210, 13, "절취선 + 구강검진 확인서"),
    (211, 230, 14, "확인서 양식"),
]

word_labels = [-1] * len(page_spans)
for start, end, sid, note in sentence_ranges:
    for i in range(start, min(end + 1, len(word_labels))):
        word_labels[i] = sid

max_sid = max(sid for _, _, sid, _ in sentence_ranges)
UNLABELED_ID = max_sid + 1
word_labels = [w if w >= 0 else UNLABELED_ID for w in word_labels]

print(f"라벨링 완료")
print(f"  총 토큰: {len(word_labels)}")
print(f"  의미 묶음: {len(set(word_labels))}개\n")
for sid, cnt in sorted(Counter(word_labels).items()):
    note = next((n for _s, _e, _sid, n in sentence_ranges if _sid == sid), "미라벨")
    print(f"  id={sid:2d}: {cnt:3d}개  — {note}")

## 8. Mini Fine-tune — 원본 순서 (B안 기본)

In [ ]:
encoded = to_layoutxlm_inputs(page_image, page_spans, word_labels=word_labels)
inputs = {k: v.to(device) for k, v in encoded.items()}

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

print("=== Mini fine-tune (원본 순서, 1 page overfit, 100 epoch) ===")
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(**inputs)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    if epoch % 10 == 0 or epoch == 99:
        print(f"  Epoch {epoch+1:3d}: loss = {loss.item():.4f}")

## 9. 원본 순서 추론 — 정확도 + sentence_list

In [ ]:
from collections import defaultdict

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
predictions = outputs.logits.argmax(-1)[0].tolist()

word_ids = encoded.word_ids()
word_preds = []
for w_idx in range(len(page_spans)):
    tok_preds = [predictions[t] for t, w in enumerate(word_ids) if w == w_idx]
    if tok_preds:
        word_preds.append(Counter(tok_preds).most_common(1)[0][0])
    else:
        word_preds.append(-1)

correct = sum(1 for p, g in zip(word_preds, word_labels) if p == g)
print(f"=== 정확도 (원본 순서 overfit) ===")
print(f"  {correct} / {len(word_preds)} = {correct/len(word_preds)*100:.1f}%")

groups = defaultdict(list)
first_app = {}
for idx, (span, sid) in enumerate(zip(page_spans, word_preds)):
    groups[sid].append(span)
    first_app.setdefault(sid, idx)
sorted_sids = sorted(groups.keys(), key=lambda s: first_app[s])

print(f"\n=== sentence_list ({len(sorted_sids)}개 묶음) ===")
for sid in sorted_sids:
    text = " ".join(s.text for s in groups[sid])
    note = next((n for _s, _e, _sid, n in sentence_ranges if _sid == sid), "")
    print(f"\n[묶음 id={sid}]  ({note})")
    print(f"  {text[:200]}")

## 10. 핵심 — 셔플 검증 (LayoutXLM 진짜 능력)

**LiLT는 셔플 시 96% → 43% 폭락** → bbox 좌표만으로 column 학습 못 함.

LayoutXLM은 **image stream까지** 활용 가능 → 셔플해도 시각 신호로 column 분리 가능한지 검증.

In [ ]:
import random

# 1) 셔플 — bbox, image, 라벨은 그대로
indices = list(range(len(page_spans)))
random.seed(42)
random.shuffle(indices)
shuffled_spans = [page_spans[i] for i in indices]
shuffled_labels = [word_labels[i] for i in indices]

print(f"셔플 전 첫 10 토큰: {[s.text for s in page_spans[:10]]}")
print(f"셔플 후 첫 10 토큰: {[s.text for s in shuffled_spans[:10]]}")
print()

# 2) 모델 재초기화
model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=MAX_SENT_ID,
    id2label={i: f"SENT_{i}" for i in range(MAX_SENT_ID)},
    label2id={f"SENT_{i}": i for i in range(MAX_SENT_ID)},
).to(device)

# 3) 셔플된 순서로 학습 (image + bbox만이 의미 신호)
encoded_sh = to_layoutxlm_inputs(page_image, shuffled_spans, word_labels=shuffled_labels)
inputs_sh = {k: v.to(device) for k, v in encoded_sh.items()}

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)

print("=== 셔플된 순서로 학습 (image + bbox만이 의미 신호) ===")
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(**inputs_sh)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    if epoch % 10 == 0 or epoch == 99:
        print(f"  Epoch {epoch+1:3d}: loss = {loss.item():.4f}")

In [ ]:
# 4) 원본 순서로 추론
encoded_orig = to_layoutxlm_inputs(page_image, page_spans)
inputs_orig = {k: v.to(device) for k, v in encoded_orig.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs_orig)
predictions = outputs.logits.argmax(-1)[0].tolist()

word_ids = encoded_orig.word_ids()
word_preds = []
for w_idx in range(len(page_spans)):
    tok_preds = [predictions[t] for t, w in enumerate(word_ids) if w == w_idx]
    if tok_preds:
        word_preds.append(Counter(tok_preds).most_common(1)[0][0])
    else:
        word_preds.append(-1)

correct = sum(1 for p, g in zip(word_preds, word_labels) if p == g)
acc = correct / len(word_preds) * 100
print(f"=== 정확도 (셔플 학습 → 원본 순서 추론) ===")
print(f"  {correct} / {len(word_preds)} = {acc:.1f}%")
print(f"  LiLT 셔플 결과: 43.3%  ← 비교 기준")

# 진심담은 vs 청담i 분리 검증
print(f"\n=== 진심담은(id=10) vs 청담i(id=11) 분리 검증 ===")
id10_texts = [page_spans[i].text for i, p in enumerate(word_preds) if p == 10]
id11_texts = [page_spans[i].text for i, p in enumerate(word_preds) if p == 11]
print(f"id=10 묶음 ({len(id10_texts)}개): {' '.join(id10_texts)[:250]}")
print()
print(f"id=11 묶음 ({len(id11_texts)}개): {' '.join(id11_texts)[:250]}")

## 11. 결과 해석

### LiLT 셔플 결과 (비교 기준)
- 정확도 **43.3%**
- 진심담은 / 청담i 완전 섞임
- → bbox 좌표만으론 layout 학습 불가능 (단어 surface로만 분류)

### LayoutXLM 셔플 결과 — 가능한 시나리오

| 정확도 | 의미 |
|---|---|
| **80%+ AND 두 묶음 분리** | image stream이 진짜 layout 학습 — **자체화 길 OK** |
| 60~80% | 일부 layout 학습 — 데이터 더 필요 |
| 43% 수준 (LiLT와 유사) | image도 부족 — 1장 학습으로는 불가능. 대량 학습 필요 |

### 다음 단계

- LayoutXLM 셔플도 80%+ → 본격 라벨링 시작
- LayoutXLM 셔플도 50% 이하 → 1장 학습 자체의 한계 → 데이터 양 또는 다른 접근 검토